# step_combined

> Phase 2 combined step renderer: dual-column layout for Segment & Align

In [ ]:
#| default_exp combined.step_combined

In [ ]:
#| export
from typing import Any, List

from fasthtml.common import Div, H2, P, Span, Input, Button

# DaisyUI components
from cjm_fasthtml_daisyui.components.data_display.badge import badge, badge_styles, badge_sizes
from cjm_fasthtml_daisyui.components.feedback.alert import alert, alert_colors
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui, text_dui, border_dui, ring_dui
from cjm_fasthtml_daisyui.utilities.border_radius import border_radius

# Tailwind utilities
from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.sizing import w, h, min_h
from cjm_fasthtml_tailwind.utilities.typography import font_size, font_weight, uppercase, tracking
from cjm_fasthtml_tailwind.utilities.layout import overflow, position, inset, display_tw
from cjm_fasthtml_tailwind.utilities.borders import border
from cjm_fasthtml_tailwind.utilities.effects import opacity, ring
from cjm_fasthtml_tailwind.utilities.transitions_and_animation import transition, duration
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import (
    flex_display, flex_direction, justify, items, gap, grow
)
from cjm_fasthtml_tailwind.core.base import combine_classes

# Interaction context
from cjm_fasthtml_interactions.core.context import InteractionContext

# Card stack library
from cjm_fasthtml_card_stack.components.controls import render_width_slider
from cjm_fasthtml_card_stack.components.states import render_loading_state
from cjm_fasthtml_card_stack.core.constants import DEFAULT_VISIBLE_COUNT, DEFAULT_CARD_WIDTH

# Local imports
from cjm_fasthtml_workflow_transcript_decomp.core.html_ids import StructureDecompHtmlIds
from cjm_fasthtml_workflow_transcript_decomp.decomposition.models import WorkingSegment, DecompUrls
from cjm_fasthtml_workflow_transcript_decomp.alignment.models import VADChunk, AlignmentUrls

# Decomposition state getters
from cjm_fasthtml_workflow_transcript_decomp.decomposition.components.helpers import (
    _is_initialized, _get_segments, _get_focused_index, _get_history,
    _get_visible_count, _get_card_width,
)

# Decomposition composable renderers
from cjm_fasthtml_workflow_transcript_decomp.decomposition.components.step_renderer import (
    render_decomp_column_body, render_toolbar, render_decomp_stats,
    render_decomp_footer_content, render_decomp_mini_stats_text,
)

# Alignment state getters
from cjm_fasthtml_workflow_transcript_decomp.alignment.components.helpers import (
    _is_alignment_initialized, _get_vad_chunks, _get_focused_chunk_index,
    _get_alignment_visible_count, _get_alignment_card_width, _get_media_path,
)

# Alignment composable renderers
from cjm_fasthtml_workflow_transcript_decomp.alignment.components.step_renderer import (
    render_align_column_body, render_align_mini_stats_text,
)
from cjm_fasthtml_workflow_transcript_decomp.alignment.components.card_stack_config import (
    ALIGN_CS_IDS,
)

# Shared keyboard config (combined-level)
from cjm_fasthtml_workflow_transcript_decomp.combined.keyboard_config import (
    build_combined_kb_system,
    render_keyboard_hints_collapsible, generate_zone_change_js,
    SWITCH_CHROME_BTN_ID,
)
from cjm_fasthtml_workflow_transcript_decomp.decomposition.components.card_stack_config import (
    DECOMP_CS_CONFIG, DECOMP_CS_IDS,
)

# Debug flag for combined step rendering tracing (set False in production)
DEBUG_COMBINED_RENDER = True

## Column Headers with Mini-Stats

Each column has a header showing its title and a badge area for compact stats
that remain visible regardless of which column is active.

In [ ]:
#| export
def _render_column_header(
    title:str,  # Column title (e.g., "Text Decomposition")
    stats_id:str,  # HTML ID for the mini-stats badge area
    header_id:str,  # HTML ID for the column header container
    initial_text:str="--",  # Initial text for the mini-stats badge
) -> Any:  # Column header component
    """Render a column header with title and mini-stats badge."""
    return Div(
        Span(
            title,
            cls=combine_classes(
                font_size.sm, font_weight.bold,
                uppercase, tracking.wide,
                text_dui.base_content.opacity(50)
            )
        ),
        Span(
            initial_text,
            id=stats_id,
            cls=combine_classes(badge, badge_styles.ghost, badge_sizes.sm)
        ),
        id=header_id,
        cls=combine_classes(
            flex_display, justify.between, items.center,
            p(3), bg_dui.base_200,
            border_dui.base_300, border.b()
        )
    )

## Mini-Stats Badge OOB

Renders the full decomposition mini-stats badge for OOB swaps from handlers.
Matches the badge styling used in `_render_column_header`.

In [ ]:
#| export
def render_decomp_mini_stats_badge(
    segments:List[WorkingSegment],  # Current segments
    oob:bool=False,  # Whether to render as OOB swap
) -> Any:  # Mini-stats badge Span
    """Render the decomposition mini-stats badge for the column header."""
    return Span(
        render_decomp_mini_stats_text(segments),
        id=StructureDecompHtmlIds.DECOMP_MINI_STATS,
        cls=combine_classes(badge, badge_styles.ghost, badge_sizes.sm),
        hx_swap_oob="true" if oob else None,
    )

In [ ]:
#| export
def render_align_mini_stats_badge(
    chunks:List[VADChunk],  # Current VAD chunks
    oob:bool=False,  # Whether to render as OOB swap
) -> Any:  # Mini-stats badge Span
    """Render the alignment mini-stats badge for the column header."""
    return Span(
        render_align_mini_stats_text(chunks),
        id=StructureDecompHtmlIds.ALIGNMENT_MINI_STATS,
        cls=combine_classes(badge, badge_styles.ghost, badge_sizes.sm),
        hx_swap_oob="true" if oob else None,
    )

## Alignment Status Indicator

Shows whether segment count matches VAD chunk count for 1:1 alignment.
Displays different states: waiting for data, mismatch with delta, or aligned.

In [ ]:
#| export
def render_alignment_status_text(
    segment_count:int,  # Number of text segments
    chunk_count:int,  # Number of VAD chunks
) -> str:  # Status message text
    """Generate alignment status message based on segment and VAD chunk counts."""
    if segment_count == 0 or chunk_count == 0:
        return "Waiting for data..."
    
    delta = segment_count - chunk_count
    
    if delta == 0:
        return f"Aligned ({segment_count})"
    elif delta > 0:
        return f"{segment_count} segments vs {chunk_count} VAD chunks (merge {delta})"
    else:
        return f"{segment_count} segments vs {chunk_count} VAD chunks (split {-delta})"


#| export
def render_alignment_status(
    segment_count:int,  # Number of text segments
    chunk_count:int,  # Number of VAD chunks
    oob:bool=False,  # Whether to render as OOB swap
) -> Any:  # Alignment status badge component
    """Render the alignment status indicator badge."""
    is_aligned = segment_count > 0 and chunk_count > 0 and segment_count == chunk_count
    
    # Choose badge style based on alignment state
    if is_aligned:
        badge_cls = combine_classes(badge, badge_sizes.sm, text_dui.success, font_weight.bold)
    elif segment_count == 0 or chunk_count == 0:
        badge_cls = combine_classes(badge, badge_styles.ghost, badge_sizes.sm)
    else:
        badge_cls = combine_classes(badge, badge_sizes.sm, text_dui.warning, font_weight.bold)
    
    return Span(
        render_alignment_status_text(segment_count, chunk_count),
        id=StructureDecompHtmlIds.ALIGNMENT_STATUS,
        cls=badge_cls,
        hx_swap_oob="true" if oob else None,
    )


# Shared footer inner content styling
_FOOTER_INNER_CLS = combine_classes(flex_display, justify.between, items.center, w.full, gap(4))


#| export
def render_footer_inner_content(
    column_footer:Any,  # Column-specific footer content (decomp or align)
    segment_count:int,  # Number of text segments
    chunk_count:int,  # Number of VAD chunks
) -> Any:  # Styled wrapper div with column footer and alignment status
    """Render the footer inner content with consistent styling.
    
    This ensures the footer layout (justify-between) is preserved across
    all OOB swaps. Both the column-specific footer content and the
    alignment status indicator are wrapped in a flex container.
    """
    return Div(
        column_footer,
        render_alignment_status(segment_count, chunk_count),
        cls=_FOOTER_INNER_CLS
    )

## Shared Chrome Containers

Stable-ID containers for keyboard hints, toolbar, controls, and footer.
When decomposition is initialized, these are populated with real content.
Otherwise, placeholder text is shown. OOB innerHTML swaps update these
when focus switches between columns or after init completes.

In [ ]:
#| export
def _placeholder(
    text:str,  # Placeholder message
) -> Any:  # Styled placeholder paragraph
    """Render a placeholder text element for uninitialized chrome containers."""
    return P(text, cls=combine_classes(font_size.sm, text_dui.base_content.opacity(50)))


def _render_shared_chrome(
    decomp_state:dict=None,  # Decomp state dict (None = show placeholders)
    align_state:dict=None,  # Alignment state dict (None = no VAD data yet)
    urls:DecompUrls=None,  # Decomp URL bundle (required when decomp_state provided)
    kb_manager:Any=None,  # Keyboard manager (required when decomp_state provided)
) -> tuple:  # (hints, toolbar, controls, footer)
    """Render shared chrome containers, populated with decomp content when initialized."""
    is_init = decomp_state is not None

    # --- Hints ---
    if is_init and kb_manager:
        hints_content = render_keyboard_hints_collapsible(kb_manager)
    else:
        hints_content = _placeholder("Keyboard hints will appear here when a column is active.")

    hints = Div(
        hints_content,
        id=StructureDecompHtmlIds.SHARED_HINTS,
        cls=str(p(2))
    )

    # --- Toolbar ---
    if is_init and urls:
        segments = [WorkingSegment.from_dict(s) for s in decomp_state.get("segments", [])]
        history = decomp_state.get("history", [])
        visible_count = decomp_state.get("visible_count", DEFAULT_VISIBLE_COUNT)
        toolbar_content = render_toolbar(
            reset_url=urls.reset,
            ai_split_url=urls.ai_split,
            undo_url=urls.undo,
            can_undo=(len(history) > 0),
            visible_count=visible_count,
        )
    else:
        toolbar_content = _placeholder("Toolbar actions will appear here based on the active column.")

    toolbar = Div(
        toolbar_content,
        id=StructureDecompHtmlIds.SHARED_TOOLBAR,
        cls=str(p(2))
    )

    # --- Controls ---
    if is_init:
        card_width = decomp_state.get("card_width", DEFAULT_CARD_WIDTH)
        controls_content = render_width_slider(DECOMP_CS_CONFIG, DECOMP_CS_IDS, card_width=card_width)
    else:
        controls_content = _placeholder("Column-specific controls will appear here.")

    controls = Div(
        controls_content,
        id=StructureDecompHtmlIds.SHARED_CONTROLS,
        cls=str(p(2))
    )

    # --- Footer with alignment status ---
    segment_count = len(decomp_state.get("segments", [])) if decomp_state else 0
    chunk_count = len(align_state.get("vad_chunks", [])) if align_state else 0
    
    if is_init:
        segments = [WorkingSegment.from_dict(s) for s in decomp_state.get("segments", [])]
        focused_index = decomp_state.get("focused_index", 0)
        column_footer = render_decomp_footer_content(segments, focused_index)
    else:
        column_footer = _placeholder("Footer with progress and timestamp details will appear here.")

    footer_inner = render_footer_inner_content(column_footer, segment_count, chunk_count)

    footer = Div(
        footer_inner,
        id=StructureDecompHtmlIds.SHARED_FOOTER,
        cls=combine_classes(
            p(1), bg_dui.base_100,
            border_dui.base_300, border.t(),
            flex_display, justify.center, items.center
        )
    )

    return hints, toolbar, controls, footer

## Column Containers

Left column (60%, text decomposition) and right column (40%, VAD alignment).
Each column has a stable HTML ID, a header with mini-stats, and a content area.
When initialized, columns render the card stack viewport.
Otherwise they show a loading state with auto-trigger for initialization.
Columns stack vertically below the `lg` breakpoint.

In [ ]:
#| export
# Shared column styling (reused by init handler for outerHTML swap)
_DECOMP_COLUMN_CLS = combine_classes(
    w.full, w('[60%]').lg,
    min_h(0),
    flex_display, flex_direction.col,
    bg_dui.base_100, border_dui.base_300, border(1),
    border_radius.box,
    overflow.hidden,
    transition.all, duration._200,
)


def _render_decomp_column(
    is_active:bool=True,  # Whether this column is initially active
    column_body:Any=None,  # Pre-rendered column body (None = not initialized)
    mini_stats_text:str="--",  # Mini-stats badge text
    init_url:str="",  # URL for auto-trigger initialization
) -> Any:  # Left column component
    """Render the left decomposition column."""
    active_cls = combine_classes(ring(1), ring_dui.primary) if is_active else str(opacity(70))

    # Content area: initialized viewport or loading + auto-trigger
    if column_body is not None:
        content = column_body
    else:
        content = Div(
            render_loading_state(DECOMP_CS_IDS, message="Initializing segments..."),
            # Auto-trigger initialization on load
            Div(
                hx_post=init_url,
                hx_trigger="load",
                hx_target=StructureDecompHtmlIds.as_selector(
                    StructureDecompHtmlIds.DECOMP_COLUMN_CONTENT
                ),
                hx_swap="outerHTML"
            ) if init_url else None,
            id=StructureDecompHtmlIds.DECOMP_COLUMN_CONTENT,
            cls=combine_classes(grow(), overflow.hidden, flex_display, flex_direction.col, p(4))
        )

    return Div(
        _render_column_header(
            title="Text Decomposition",
            stats_id=StructureDecompHtmlIds.DECOMP_MINI_STATS,
            header_id=StructureDecompHtmlIds.DECOMP_COLUMN_HEADER,
            initial_text=mini_stats_text,
        ),
        content,
        id=StructureDecompHtmlIds.DECOMP_COLUMN,
        cls=combine_classes(_DECOMP_COLUMN_CLS, active_cls)
    )

In [ ]:
#| export
# Shared alignment column styling (reused by init handler for outerHTML swap)
_ALIGNMENT_COLUMN_CLS = combine_classes(
    w.full, w('[40%]').lg,
    min_h(0),
    flex_display, flex_direction.col,
    bg_dui.base_100, border_dui.base_300, border(1),
    border_radius.box,
    overflow.hidden,
    transition.all, duration._200,
)


def _render_alignment_column(
    is_active:bool=False,  # Whether this column is initially active
    column_body:Any=None,  # Pre-rendered column body (None = not initialized)
    mini_stats_text:str="--",  # Mini-stats badge text
    init_url:str="",  # URL for auto-trigger initialization
) -> Any:  # Right column component
    """Render the right alignment column."""
    active_cls = combine_classes(ring(1), ring_dui.secondary) if is_active else str(opacity(70))

    # Content area: initialized viewport or loading + auto-trigger
    if column_body is not None:
        content = column_body
    else:
        content = Div(
            render_loading_state(ALIGN_CS_IDS, message="Analyzing audio..."),
            # Auto-trigger initialization on load
            Div(
                hx_post=init_url,
                hx_trigger="load",
                hx_target=StructureDecompHtmlIds.as_selector(
                    StructureDecompHtmlIds.ALIGNMENT_COLUMN_CONTENT
                ),
                hx_swap="outerHTML"
            ) if init_url else None,
            id=StructureDecompHtmlIds.ALIGNMENT_COLUMN_CONTENT,
            cls=combine_classes(grow(), min_h(0), overflow.hidden, flex_display, flex_direction.col, p(4))
        )

    return Div(
        _render_column_header(
            title="VAD Alignment",
            stats_id=StructureDecompHtmlIds.ALIGNMENT_MINI_STATS,
            header_id=StructureDecompHtmlIds.ALIGNMENT_COLUMN_HEADER,
            initial_text=mini_stats_text,
        ),
        content,
        id=StructureDecompHtmlIds.ALIGNMENT_COLUMN,
        cls=combine_classes(_ALIGNMENT_COLUMN_CLS, active_cls)
    )

## Keyboard System Container

Stable container for keyboard navigation system (script, hidden inputs, action buttons).
This container lives outside both column bodies so it doesn't get destroyed/recreated
when columns are swapped. OOB innerHTML swaps update its content when the combined
KB system is built (after both columns are initialized in 4d).

In [ ]:
#| export
def _render_keyboard_system_container(
    kb_system:Any=None,  # Rendered keyboard system (None = empty container)
    oob:bool=False,  # Whether to render as OOB swap
) -> Any:  # Div with id=KEYBOARD_SYSTEM containing KB elements
    """Render stable container for keyboard navigation system elements."""
    if DEBUG_COMBINED_RENDER:
        print(f"[COMBINED_RENDER] _render_keyboard_system_container called, kb_system={kb_system is not None}")

    if kb_system is not None:
        content = (kb_system.script, kb_system.hidden_inputs, kb_system.action_buttons)
    else:
        content = ()

    return Div(
        *content,
        id=StructureDecompHtmlIds.KEYBOARD_SYSTEM,
        hx_swap_oob="true" if oob else None,
    )

## Main Combined Step Renderer

Top-level step renderer for the merged Phase 2. Composes the shared chrome,
dual-column layout, and a hidden input tracking the active column.

In [ ]:
#| export
def render_combined_step(
    ctx:InteractionContext,  # Interaction context with state and data
    decomp_urls:DecompUrls=None,  # URL bundle for decomposition routes
    align_urls:AlignmentUrls=None,  # URL bundle for alignment routes
    switch_chrome_url:str="",  # URL for chrome switching route
) -> Any:  # FastHTML component with full dual-column layout
    """Render Phase 2: Combined Segment & Align step with dual-column layout."""
    if DEBUG_COMBINED_RENDER:
        print("[COMBINED_RENDER] render_combined_step called")

    decomp_urls = decomp_urls or DecompUrls()

    # Check initialization states
    is_decomp_init = _is_initialized(ctx)
    is_align_init = _is_alignment_initialized(ctx)

    if DEBUG_COMBINED_RENDER:
        print(f"[COMBINED_RENDER] is_decomp_init: {is_decomp_init}")
        print(f"[COMBINED_RENDER] is_align_init: {is_align_init}")

    # Variables for KB system
    kb_manager = None
    kb_system = None
    zone_change_js = None

    # Get raw state dicts for shared chrome
    decomp_state_dict = ctx.state.get("step_states", {}).get("decomposition", {}) if is_decomp_init else None
    align_state_dict = ctx.state.get("step_states", {}).get("alignment", {})

    if is_decomp_init:
        # Build decomp pieces from state
        segments = _get_segments(ctx)
        focused_index = _get_focused_index(ctx)
        history = _get_history(ctx)
        visible_count = _get_visible_count(ctx, default=DEFAULT_VISIBLE_COUNT)
        card_width = _get_card_width(ctx, default=DEFAULT_CARD_WIDTH)

        # Always build combined KB system with both zones
        # (alignment zone will have no items until alignment init completes)
        if align_urls:
            kb_manager, kb_system = build_combined_kb_system(decomp_urls, align_urls)
            zone_change_js = generate_zone_change_js(switch_chrome_url)
            if DEBUG_COMBINED_RENDER:
                print(f"[COMBINED_RENDER] built combined kb_system with both zones")

        # Render column body with viewport (kb_system=None, KB is in stable container)
        column_body = render_decomp_column_body(
            segments=segments,
            focused_index=focused_index,
            visible_count=visible_count,
            card_width=card_width,
            urls=decomp_urls,
            kb_system=None,  # KB system is in stable container, not column body
        )
        mini_stats_text = render_decomp_mini_stats_text(segments)

        # Shared chrome with real decomp content (includes alignment status)
        hints, toolbar, controls, footer = _render_shared_chrome(
            decomp_state=decomp_state_dict,
            align_state=align_state_dict,
            urls=decomp_urls,
            kb_manager=kb_manager,
        )

        # Left column with real content
        decomp_col = _render_decomp_column(
            is_active=True,
            column_body=column_body,
            mini_stats_text=mini_stats_text,
        )
    else:
        # Shared chrome with placeholders (includes alignment status)
        hints, toolbar, controls, footer = _render_shared_chrome(
            align_state=align_state_dict,
        )

        # Left column with loading state + auto-trigger
        decomp_col = _render_decomp_column(
            is_active=True,
            init_url=decomp_urls.init,
        )

    # --- Alignment column ---
    if is_align_init and align_urls:
        # Render alignment column body from state
        chunks = _get_vad_chunks(ctx)
        align_focused = _get_focused_chunk_index(ctx)
        align_visible = _get_alignment_visible_count(ctx, default=5)
        align_width = _get_alignment_card_width(ctx, default=40)
        # Web Audio API handles accurate seeking
        media_path = _get_media_path(ctx)

        if DEBUG_COMBINED_RENDER:
            print(f"[COMBINED_RENDER] media_path: {media_path}")

        align_body = render_align_column_body(
            chunks=chunks,
            focused_index=align_focused,
            visible_count=align_visible,
            card_width=align_width,
            urls=align_urls,
            kb_system=None,  # KB system is in stable container
            media_path=media_path,
        )
        align_mini_text = render_align_mini_stats_text(chunks)

        align_col = _render_alignment_column(
            is_active=False,
            column_body=align_body,
            mini_stats_text=align_mini_text,
        )
    elif align_urls and align_urls.init:
        # Show loading state with auto-trigger
        align_col = _render_alignment_column(
            is_active=False,
            init_url=align_urls.init,
        )
    else:
        # No alignment URLs — show column with default placeholder
        align_col = _render_alignment_column(is_active=False)

    # Hidden input tracking active column (decomp or align)
    active_column_input = Input(
        type="hidden",
        id=StructureDecompHtmlIds.ACTIVE_COLUMN_INPUT,
        name="active_column",
        value="decomp",
    )

    # Stable keyboard system container (outside columns)
    kb_container = _render_keyboard_system_container(kb_system=kb_system)

    # Hidden button for chrome swap HTMX trigger (clicked by zone change JS)
    chrome_switch_btn = None
    if switch_chrome_url and align_urls:
        chrome_switch_btn = Button(
            id=SWITCH_CHROME_BTN_ID,
            cls=str(display_tw.hidden),
            hx_post=switch_chrome_url,
            hx_include=f"#{StructureDecompHtmlIds.ACTIVE_COLUMN_INPUT}",
            hx_swap="none",
        )

    if DEBUG_COMBINED_RENDER:
        print(f"[COMBINED_RENDER] kb_container rendered with kb_system={kb_system is not None}")
        print(f"[COMBINED_RENDER] zone_change_js present: {zone_change_js is not None}")
        print(f"[COMBINED_RENDER] chrome_switch_btn present: {chrome_switch_btn is not None}")

    return Div(
        # Header
        Div(
            H2("Segment & Align", cls=combine_classes(font_size._3xl, font_weight.bold)),
            P(
                "Decompose text into segments and align with audio timestamps.",
                cls=combine_classes(text_dui.base_content.opacity(70), m.b(2))
            ),
        ),

        # Shared keyboard hints
        hints,

        # Shared toolbar and controls
        toolbar,
        controls,

        # Dual-column content area
        Div(
            decomp_col,
            align_col,
            cls=combine_classes(
                grow(),
                min_h(0),
                flex_display,
                flex_direction.col,
                flex_direction.row.lg,
                gap(4),
                overflow.hidden,
                p(1),
            )
        ),

        # Shared footer
        footer,

        # Stable keyboard system container (outside columns, won't be swapped)
        kb_container,

        # Zone change JavaScript (when align_urls available)
        zone_change_js,

        # Hidden chrome switch button (for HTMX-triggered chrome swaps)
        chrome_switch_btn,

        # Hidden active column state
        active_column_input,

        id=StructureDecompHtmlIds.DECOMP_CONTAINER,
        cls=combine_classes(
            w.full, h.full,
            flex_display, flex_direction.col,
            p(4), p.x(2), p.b(0)
        )
    )

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()